In [ ]:
from typing import Iterable
from itertools import chain as iterchain, combinations as itercomb
from collections import deque

iterflat = iterchain.from_iterable

In [ ]:
from board import DIGITS, Cell, Node, Board
from utils import countfinals
from analytics import Locality, Target, Link, HLink, SLink, Chain, validate, all_visible, are_nandable, Node_has
from solving import orchestrator, solver, Resolution, Resolving, Resolver

In [ ]:
def fillempty(node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

In [ ]:
async def solve_silent(initial: Board, *resolvers: Resolver):
    result = initial
    async for _, _, result in solver(initial, orchestrator(initial, *resolvers)):
        pass
    return result

In [ ]:
async def solve_logging(initial: Board, *resolvers: Resolver):
    result = initial
    async for resolver, resolution, result in solver(initial, orchestrator(initial, *resolvers)):
        print(resolver.__name__, end=": ")
        if resolution.castaways:
            print("-={", " ".join(map(str, resolution.castaways)), "}", end=" ")
        if resolution.finals:
            print(":={", " ".join(map(str, resolution.finals)), "}", end=" ")
        if resolution.highlights:
            if "zone" in resolution.highlights:
                print("@", resolution.highlights["zone"], end=" ")
            print("#", end=" ")
            if "anchors" in resolution.highlights:
                print(" ".join(map(str, resolution.highlights["anchors"])), end=" ")
            if "chain" in resolution.highlights:
                print(resolution.highlights["chain"], end=" ")
        print()
    print(validate(result))
    return result

## Basic

Singles in localities and their contra-neighbors


In [ ]:
def cleanup(board: Board) -> Resolving:
    """Removing drafts contradicting with neighbouring finals"""

    for zone in Locality.all():
        finalborhood = list(filter(Node.is_final, zone.neighborhood(board)))
        draftborhood = list(filter(Node.is_draft, zone.neighborhood(board)))
        for finode in finalborhood:
            dig = finode.cell.final
            assert dig is not None
            contras = list(filter(Node_has(dig), draftborhood))
            if contras:
                yield Resolution(
                    castaways=set(Target(n.loc, dig) for n in contras),
                    highlights={"anchors": {Target(finode.loc, dig)}, "zone": zone},
                )

In [ ]:
def singles(board: Board) -> Resolving:
    """Isolate singular digits in localities"""

    for zone in Locality.all():
        draftborhood = list(filter(Node.is_draft, zone.neighborhood(board)))
        for dig in DIGITS:
            family = list(filter(Node_has(dig), draftborhood))
            if len(family) == 1 and len(family[0]) > 1:
                lonesome = Target(family[0].loc, dig)
                yield Resolution(
                    finals={lonesome},
                    highlights={"zone": zone},
                )

## Multiples

Combos of N digits

### open

Some n cells (within locality) contains only n-combo // the digits may be in other cells

=> remove the digits of the combo from all other cells

### hidden

Some n-combo contained in only n cells (within locality) // along other drafts

=> remove all other drafts from the cells => it becomes open


In [ ]:
def all_combos(m: int):
    return map(set[int], itercomb(DIGITS, m))

In [ ]:
def openmults(board: Board, mult: int) -> Resolving:
    """Clar out spoiling neighbours of open multiples in each zone"""
    for zone in Locality.all():
        draftborhood = set(filter(Node.is_draft, zone.neighborhood(board)))
        for combo in all_combos(mult):
            # all nodes containing only the combo
            habitat = set(filter(lambda n: n.cell <= combo, draftborhood))
            # all other neighbors containing some combo digits
            spoilers = tuple(filter(lambda n: n.cell & combo, draftborhood - habitat))
            if len(habitat) == mult and len(spoilers) > 0:
                # TODO: use MultiTarget
                castaways = set(Target(n.loc, d) for n in spoilers for d in n.cell & combo)
                anchors = set(Target(n.loc, d) for n in habitat for d in n.cell & combo)
                yield Resolution(
                    castaways,
                    highlights={"anchors": anchors, "zone": zone},
                )


def openmults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return openmults(board, mult)

    resolver.__name__ = f"openmults[{mult}]"
    return resolver

In [ ]:
def unhidemults(board: Board, mult: int) -> Resolving:
    """Clean up cellmates of hidden multiples"""
    for zone in Locality.all():
        draftborhood = tuple(filter(Node.is_draft, zone.neighborhood(board)))
        for combo in all_combos(mult):
            # all nodes containing the combo (+something else)
            habitat = tuple(filter(lambda n: n.cell >= combo, draftborhood))
            # all combo digits in the nodes
            habitants = tuple(iterflat(map(lambda n: n.cell & combo, habitat)))
            if len(habitat) == mult and len(habitants) == mult:
                # nodes with other digits
                spoiled = tuple(filter(lambda n: n.cell - combo, habitat))
                if len(spoiled):
                    # TODO: use MultiTarget
                    castaways = set(Target(n.loc, d) for n in spoiled for d in n.cell - combo)
                    anchors = set(Target(n.loc, d) for n in habitat for d in n.cell & combo)
                    yield Resolution(
                        castaways,
                        highlights={"anchors": anchors, "zone": zone},
                    )


def unhidemults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return unhidemults(board, mult)

    resolver.__name__ = f"unhidemults[{mult}]"
    return resolver

## Links/Chains

Links represent XOR or NAND relations between drafts.

- XOR $\veebar$ corresponds to "each digit appears only once in a locality"
- NAND $\barwedge$ corrsponds to "each locality contains only different digits"

(or vise versa, I dunno)

### weak/soft links

Represent NAND relation $\barwedge$

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts (of different digits) in a cell

Note: The criteria are totally independent of board content (calculating from locations only)

Visibility = soft-linkability

### strong/hard links

Represent XOR relation $\veebar$

Criteria:

- only 2 drafts of same digit in a locality
- only 2 drafts (of different digits) in a cell

### alterating chains

Chains constituted of `~ hard ~ soft ~` and `~ soft ~ hard ~`

Lemma1: $(X \barwedge A) \cdot (A \veebar B) \cdot (B \barwedge X) \Rightarrow \neg X$

Meaning: all draft visible (soft-linkable) from some XORed points, are all invalid

Lemma2: $(X \veebar A) \cdot (A \barwedge B) \cdot (B \veebar Y) \Rightarrow (X \veebar Y)$

Meaning: a ALC (of any length) with hard edges behaves as if its edges are hard-linked

### open ALC

ALC with hard links at its edges: `X ~ hard ~ ... ~ hard ~ Y`

Rule: invalidate all drafts visible from both edges of such chain

### loop ALC

ALC with connected edges: `X ~ hard ~ ... ~ soft ~ X`

Rule: invalidate all draft visible from each soft-link in the chain


In [ ]:
def search_hard(board: Board) -> Iterable[HLink]:
    """Search for all hard links"""

    def scan_cell(node: Node):
        if len(node.cell) == 2:
            d1, d2 = node.cell
            yield HLink((Target(node.loc, d1), Target(node.loc, d2)))

    def scan_locality(zone: Locality):
        draftborhood = tuple(filter(Node.is_draft, zone.neighborhood(board)))
        for d in DIGITS:
            family = tuple(filter(Node_has(d), draftborhood))
            if len(family) == 2:
                n1, n2 = family
                yield HLink((Target(n1.loc, d), Target(n2.loc, d)))

    for node in filter(Node.is_draft, board):
        yield from scan_cell(node)

    for zone in Locality.all():
        yield from scan_locality(zone)

In [ ]:
def search_chains(links: Iterable[HLink], max_length: int) -> Iterable[Chain]:
    """Search for all chains"""

    # breadth-first graph search
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # queue
    explored = set[Chain]()
    while frontier:
        chain = frontier.popleft()
        if len(chain) > 2:
            # yielding all found chains
            yield chain
        explored.add(chain)
        if len(chain) < max_length:
            frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)


def expand_alc(chain: Chain, links: Iterable[HLink]) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""
    e1, e2 = chain.edges

    # closing loop
    if len(chain) > 2 and isinstance(chain[0], HLink) and isinstance(chain[-1], HLink) and are_nandable(e1, e2):
        yield Chain.extend(chain, SLink((e2, e1)))

    anchors: set[Target] = chain.anchors()

    def noncycling(lnk: Link):
        return lnk[0] not in anchors and lnk[1] not in anchors

    for link in filter(noncycling, links):
        x1, x2 = link
        if are_nandable(e2, x1):
            yield Chain.extend(chain, SLink((e2, x1)), link)
        if are_nandable(e2, x2):
            yield Chain.extend(chain, SLink((e2, x2)), link.reversed())

In [ ]:
def match_loop(chain: Chain):
    """ALC loop"""
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def match_rope(chain: Chain):
    """ALC with matching edges"""
    e1, e2 = chain.edges
    return len(chain) > 2 and len(chain) % 2 == 1 and e1.dig == e2.dig

In [ ]:
def scan_visible(board: Board, e1: Target, e2: Target, anchors: set[Target]) -> Iterable[Target]:
    for node in filter(Node.is_draft, board.slice(all_visible(e1.loc, e2.loc))):
        for d in node.cell:
            trg = Target(node.loc, d)
            if trg not in anchors and are_nandable(trg, e1) and are_nandable(trg, e2):
                yield trg


def resolve_loop(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible for each soft link"""
    anchors = chain.anchors()

    for link in filter(lambda lnk: isinstance(lnk, SLink), chain):
        t1, t2 = link
        spoilers = set(scan_visible(board, t1, t2, anchors))
        if len(spoilers):
            yield Resolution(
                spoilers,
                highlights={"anchors": {t1, t2}, "chain": chain},
            )


def resolve_rope(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible from both edges"""
    anchors = chain.anchors()
    e1, e2 = chain.edges

    spoilers = set(scan_visible(board, e1, e2, anchors))
    if len(spoilers):
        yield Resolution(
            spoilers,
            highlights={"anchors": {e1, e2}, "chain": chain},
        )


def resolve_chains(current: Board, max_length: int) -> Resolving:
    links = set(search_hard(current))
    for chain in search_chains(links, max_length):  # NB: all found chains
        if match_loop(chain):
            yield from resolve_loop(current, chain)
        elif match_rope(chain):
            yield from resolve_rope(current, chain)


def chains_(max_length: int):
    def resolver(board: Board) -> Resolving:
        return resolve_chains(board, max_length)

    resolver.__name__ = f"chains[max={max_length}]"
    return resolver

## A puzzle


In [ ]:
from utils import parse


puzzle = parse("""
753......
.....7.9.
..65.....
.....68..
...7.96..
.94....3.
...84.1..
.2..5..6.
5........
""")


In [ ]:
puzzle = Board.transform(puzzle, fillempty)

In [ ]:
puzzle = await solve_silent(puzzle, cleanup, singles)

In [ ]:
puzzle = await solve_logging(
    puzzle,
    cleanup,
    singles,
    openmults_(2),
    unhidemults_(2),
    openmults_(3),
    unhidemults_(3),
    openmults_(4),
    unhidemults_(4),
    openmults_(5),
    unhidemults_(5),
    chains_(8),
)

### GUI


In [ ]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-input-color: var(--vscode-editor-foreground);
    --jp-widgets-input-background-color: var(--vscode-editor-background);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.jupyter-widgets input {
   background-color: var(--jp-widgets-input-background-color);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [ ]:
import asyncio
from collections import Counter
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display


from traitlets import HasTraits, Instance, Set, Unicode, observe, Enum, Bool, Dict
from canvas import SudokuCanvas

In [ ]:
"""GUI meta-widget"""

debug_view = w.Output()


def click_future(button: w.Button) -> asyncio.Future[bool]:
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


LINK_STYLES = {
    Link: "SOLID",
    HLink: "HARD",
    SLink: "SOFT",
}


class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Enum(["INCOMPLETE", "SOLVED", "BROKEN"])
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Target))
    finals = Set(Instance(Target))
    anchors = Set(Instance(Target))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()
        self._counters = {
            dig: w.Label(
                "...",
                layout=dict(width="auto"),
                # style=dict(background="transparent"),
            )
            for dig in DIGITS
        }
        self._counter_total = w.Label(
            "...",
            layout=dict(width="auto"),
            # style=dict(background="transparent"),
        )
        self._status = w.Label(layout=dict(width="auto"), style=dict(text_color="white"))
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._counter_total, self._status],
                    layout=dict(align_items="stretch", width="6em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self.status = validate(self.puzzle)
        self.counters = countfinals(self.puzzle)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "transparent"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            self._counters[dig].value = f"({dig}): {cnt}"
            self._counters[dig].style.background = "var(--jp-success-color0)" if cnt == 9 else ""
        total = counters.total()
        self._counter_total.value = f"Total: {total}"
        self._counter_total.style.background = "var(--jp-success-color0)" if total == 81 else ""

    @observe("targets", "finals", "anchors", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for lnk in self.links:
                t1, t2 = lnk
                self._canvas.highlight_link(t1.loc, t1.dig, t2.loc, t2.dig, style=LINK_STYLES[lnk.__class__], color="blue")
                self._canvas.highlight_segment(t1.loc, t1.dig, color="blue")
                self._canvas.highlight_segment(t2.loc, t2.dig, color="blue")

            for trg in self.anchors:
                if isinstance(trg, Target):
                    self._canvas.highlight_segment(trg.loc, trg.dig, color="cyan")

            for trg in self.anchors:
                if isinstance(trg, Target):
                    if trg.dig in self.puzzle.get(trg.loc).cell:
                        self._canvas.highlight_digit(trg.loc, trg.dig, str(trg.dig))

            for trg in self.targets:
                if isinstance(trg, Target):
                    self._canvas.highlight_segment(trg.loc, trg.dig, color="orange")

            for trg in self.finals:
                self._canvas.highlight_segment(trg.loc, trg.dig, color="purple")
                # self._canvas.highlight_digit(trg.loc, trg.dig, str(trg.dig))

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        with debug_view:
            self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    def click_continue(self) -> asyncio.Future[bool]:
        return click_future(self._continue)

    async def pause(self):
        self.paused = True
        await self.click_continue()
        self.paused = False

In [ ]:
debug_view

In [ ]:
gui = GUI()
display(gui)

In [ ]:
gui.puzzle = puzzle

In [ ]:
async def solve_ui(initial: Board, *resolvers: Resolver):
    result = initial
    gui.puzzle = result
    gui.running = True
    gui.inspecting = {r.__name__: True for r in resolvers}

    async for resolver, resolution, result in solver(initial, orchestrator(initial, *resolvers)):
        if gui.inspecting[resolver.__name__]:
            gui.resolving = f"{resolver.__name__} -{len(resolution.castaways)} ={len(resolution.finals)}"

            gui.paused = True
            render_resolution(resolution)
            await gui.click_continue()
            clear_resolution()
            gui.paused = False

            gui.puzzle = result
            await asyncio.sleep(0.2)
            gui.resolving = ""
        else:
            gui.puzzle = result

    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Resolution):
    gui.targets = res.castaways
    gui.finals = res.finals
    if res.highlights is not None:
        gui.anchors = res.highlights.get("anchors", set())
        if "chains" in res.highlights:
            gui.links = set(iterflat(res.highlights["chains"]))
        elif "links" in res.highlights:
            gui.links = res.highlights["links"]
        else:
            gui.links = set()


def clear_resolution():
    gui.targets = set()
    gui.finals = set()
    gui.anchors = set()
    gui.links = set()


In [ ]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        cleanup,
        singles,
        openmults_(2),
        unhidemults_(2),
        openmults_(3),
        unhidemults_(3),
        openmults_(4),
        unhidemults_(4),
        openmults_(5),
        unhidemults_(5),
        chains_(8),
    )
)

In [ ]:
puzzle = task.result()

In [ ]:
task.cancel()  # FIXME: this breaks everything tho

In [ ]:
async def iter_chains():
    gui.running = True
    links = set(search_hard(puzzle))
    for chain in search_chains(links, 6):
        # if match_loop(chain) or match_rope(chain):
        if len(chain) > 3:
            chainlinks = set(chain)
            gui.links = set(chainlinks)
            gui.anchors = set(iterflat(chainlinks))
            await gui.pause()
            gui.links = set()
            gui.anchors = set()
    gui.running = False


asyncio.create_task(iter_chains())